# Redrob India Runs ATS Sandbox Demo

This Colab-safe demo runs the same ranking path on the committed sample file only. It does not embed, fetch, or require the full private candidate dataset.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/rishicodesforfun/India-runs-ats.git"
REPO_DIR = Path("/content/India-runs-ats")

if not (REPO_DIR / "phase_b" / "rank.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Using repo at {REPO_DIR}")

In [ ]:
import json
import time
from pathlib import Path

from phase_b.rank import (
    DEFAULT_COMPANY_LOOKUP_PATH,
    DEFAULT_FOUNDING_YEAR_LOOKUP_PATH,
    DEFAULT_TITLE_LOOKUP_PATH,
    rank_candidates,
    write_portal_csv,
)

sample_path = Path("data/sample/sample_candidates.json")
sample_jsonl_path = Path("/content/redrob_sample_candidates.jsonl")
output_path = Path("/content/sandbox_official_submission.csv")

with sample_path.open("r", encoding="utf-8") as handle:
    sample_candidates = json.load(handle)[:100]

with sample_jsonl_path.open("w", encoding="utf-8") as handle:
    for candidate in sample_candidates:
        handle.write(json.dumps(candidate, ensure_ascii=False) + "\n")

started = time.perf_counter()
result = rank_candidates(
    sample_jsonl_path,
    DEFAULT_COMPANY_LOOKUP_PATH,
    DEFAULT_TITLE_LOOKUP_PATH,
    DEFAULT_FOUNDING_YEAR_LOOKUP_PATH,
)
elapsed = time.perf_counter() - started
limit = min(100, len(result["ranked"]))
write_portal_csv(result["ranked"], output_path, limit=limit)

print(json.dumps({
    "sample_candidates_loaded": len(sample_candidates),
    "sample_candidates_ranked_after_honeypot": len(result["ranked"]),
    "rows_written": limit,
    "output_path": str(output_path),
    "elapsed_seconds": round(elapsed, 3),
}, indent=2))

In [ ]:
import csv

with output_path.open("r", encoding="utf-8", newline="") as handle:
    reader = csv.DictReader(handle)
    preview = [
        {"candidate_id": row["candidate_id"], "rank": row["rank"], "score": row["score"]}
        for _, row in zip(range(5), reader)
    ]

preview